# 02. PE label and prior audit

Start from the official lightweight summary tables, then compare them with the deep PESummary inspection reports. This notebook discovers candidates; it does not freeze a production label or reconstruct a prior.

In [24]:
from pathlib import Path
import json
import os
import pandas as pd

inventory_dir = Path(os.environ["GWTC_RESULTS_ROOT"]) / "pilot" / "inventory"
index_path = inventory_dir / "pe_release_index.json"
index = json.loads(index_path.read_text())
index["ready_for_label_audit"], index_path

(True,
 PosixPath('/Projects/Data/macias/924994587/GW_Project_Data/results/pilot/inventory/pe_release_index.json'))

In [25]:
candidate_rows = []
for release, payload in index["releases"].items():
    if not payload.get("present", False):
        continue
    for event_id, event in payload["events"].items():
        for row in event["rows"]:
            candidate_rows.append({"release": release, "event_id": event_id, **row})
candidates = pd.DataFrame(candidate_rows)
candidates[[
    "release", "event_id", "result_samples_key", "result_file_name",
    "mass_1_source_median", "mass_2_source_median", "redshift_median"
]] if len(candidates) else candidates

,release,event_id,result_samples_key,result_file_name,mass_1_source_median,mass_2_source_median,redshift_median
0,o4a,GW230627_015337,C00:IMRPhenomXPHM-SpinTaylor,IGWN-GWTC4p1-18965dda8_5-GW230627_015337-combi...,8.483478,5.714881,0.066302
1,o4a,GW230627_015337,C00:Mixed,IGWN-GWTC4p1-18965dda8_5-GW230627_015337-combi...,8.449413,5.742252,0.065766
2,o4a,GW230627_015337,C00:SEOBNRv5PHM,IGWN-GWTC4p1-18965dda8_5-GW230627_015337-combi...,8.422318,5.762235,0.065132
3,o4a,GW230726_002940,C00:IMRPhenomXPHM-SpinTaylor,IGWN-GWTC4p1-18965dda8_5-GW230726_002940-combi...,35.859585,27.815026,0.373798
4,o4a,GW230726_002940,C00:Mixed,IGWN-GWTC4p1-18965dda8_5-GW230726_002940-combi...,35.554629,27.895568,0.372801
5,o4a,GW230726_002940,C00:NRSur7dq4,IGWN-GWTC4p1-18965dda8_5-GW230726_002940-combi...,35.427186,27.972493,0.372840
6,o4a,GW230726_002940,C00:SEOBNRv5PHM,IGWN-GWTC4p1-18965dda8_5-GW230726_002940-combi...,35.365654,27.940824,0.371909
7,o4a,GW230814_230901,C00:IMRPhenomXO4a,IGWN-GWTC4p1-18965dda8_5-GW230814_230901-combi...,33.242359,28.295266,0.059460
8,o4a,GW230814_230901,C00:IMRPhenomXPHM-SpinTaylor,IGWN-GWTC4p1-18965dda8_5-GW230814_230901-combi...,33.753063,28.286814,0.060964
9,o4a,GW230814_230901,C00:Mixed,IGWN-GWTC4p1-18965dda8_5-GW230814_230901-combi...,33.740290,28.196569,0.062482


In [26]:
deep_rows = []
for path in sorted(inventory_dir.glob("pe_*_deep.json")):
    report = json.loads(path.read_text())
    # event_id = path.name.split("_")[1] if path.name.startswith("pe_GW") else path.stem
    event_id = "_".join(path.stem.split("_")[1:-1])
    for analysis in report.get("analyses", []):
        deep_rows.append({
            "report": path.name,
            "event_id": event_id,
            "label": analysis["label"],
            "sample_count": analysis["sample_count"],
            "parameter_count": len(analysis["parameters"]),
            "has_mass_1_source": "mass_1_source" in analysis["parameters"],
            "has_mass_2_source": "mass_2_source" in analysis["parameters"],
            "has_redshift": "redshift" in analysis["parameters"],
        })
deep_df = pd.DataFrame(deep_rows)
deep_df

,report,event_id,label,sample_count,parameter_count,has_mass_1_source,has_mass_2_source,has_redshift
0,pe_GW230627_015337_deep.json,GW230627_015337,C00:IMRPhenomXPHM-SpinTaylor,16191,149,True,True,True
1,pe_GW230627_015337_deep.json,GW230627_015337,C00:Mixed,32382,149,True,True,True
2,pe_GW230627_015337_deep.json,GW230627_015337,C00:SEOBNRv5PHM,16577,149,True,True,True


In [27]:
deep_df[deep_df["event_id"] == "GW230627_015337"]

,report,event_id,label,sample_count,parameter_count,has_mass_1_source,has_mass_2_source,has_redshift
0,pe_GW230627_015337_deep.json,GW230627_015337,C00:IMRPhenomXPHM-SpinTaylor,16191,149,True,True,True
1,pe_GW230627_015337_deep.json,GW230627_015337,C00:Mixed,32382,149,True,True,True
2,pe_GW230627_015337_deep.json,GW230627_015337,C00:SEOBNRv5PHM,16577,149,True,True,True


In [28]:
deep_report = json.loads(
    (inventory_dir / "pe_GW230627_015337_deep.json").read_text()
)

for analysis in deep_report["analyses"]:
    print("\nLABEL:", analysis["label"])
    print("sample_count:", analysis["sample_count"])
    print("parameters:")
    print(analysis["parameters"])


LABEL: C00:IMRPhenomXPHM-SpinTaylor
sample_count: 16191
parameters:
['H1_log_likelihood', 'H1_matched_filter_snr', 'H1_optimal_snr', 'H1_time', 'L1_log_likelihood', 'L1_matched_filter_snr', 'L1_optimal_snr', 'a_1', 'a_2', 'azimuth', 'chi_1_in_plane', 'chi_2_in_plane', 'chi_eff', 'chi_p', 'chirp_mass', 'chirp_mass_source', 'comoving_distance', 'cos_tilt_1', 'cos_tilt_2', 'dec', 'geocent_time', 'iota', 'log_likelihood', 'log_prior', 'luminosity_distance', 'mass_1', 'mass_1_source', 'mass_2', 'mass_2_source', 'mass_ratio', 'phase', 'phi_1', 'phi_12', 'phi_2', 'phi_jl', 'psi', 'ra', 'recalib_H1_amplitude_0', 'recalib_H1_amplitude_1', 'recalib_H1_amplitude_2', 'recalib_H1_amplitude_3', 'recalib_H1_amplitude_4', 'recalib_H1_amplitude_5', 'recalib_H1_amplitude_6', 'recalib_H1_amplitude_7', 'recalib_H1_amplitude_8', 'recalib_H1_amplitude_9', 'recalib_H1_frequency_0', 'recalib_H1_frequency_1', 'recalib_H1_frequency_2', 'recalib_H1_frequency_3', 'recalib_H1_frequency_4', 'recalib_H1_frequency_5

In [29]:
required = [
    "mass_1_source",
    "mass_2_source",
    "mass_ratio",
    "redshift",
    "luminosity_distance",
    "mass_1",
    "mass_2",
    "a_1",
    "a_2",
    "tilt_1",
    "tilt_2",
    "chi_eff",
    "chi_p",
]

for analysis in deep_report["analyses"]:
    params = set(analysis["parameters"])
    print("\n", analysis["label"])
    for p in required:
        print(f"{p:25s} {p in params}")


 C00:IMRPhenomXPHM-SpinTaylor
mass_1_source             True
mass_2_source             True
mass_ratio                True
redshift                  True
luminosity_distance       True
mass_1                    True
mass_2                    True
a_1                       True
a_2                       True
tilt_1                    True
tilt_2                    True
chi_eff                   True
chi_p                     True

 C00:Mixed
mass_1_source             True
mass_2_source             True
mass_ratio                True
redshift                  True
luminosity_distance       True
mass_1                    True
mass_2                    True
a_1                       True
a_2                       True
tilt_1                    True
tilt_2                    True
chi_eff                   True
chi_p                     True

 C00:SEOBNRv5PHM
mass_1_source             True
mass_2_source             True
mass_ratio                True
redshift                  True
luminosit

In [30]:
deep_report.keys()

dict_keys(['analyses', 'config_container_type', 'generic_hdf5', 'labels', 'path', 'pesummary_available', 'pesummary_error', 'prior_container_type'])

In [33]:
deep_report["analyses"][0].keys()

dict_keys(['label', 'parameters', 'sample_count'])

In [34]:
shallow_path = inventory_dir / "pe_GW230627_015337_shallow.json"

shallow_report = json.loads(shallow_path.read_text())

shallow_report.keys()

dict_keys(['generic_hdf5', 'labels', 'path', 'pesummary_available', 'pesummary_error'])

In [36]:
nodes = shallow_report["generic_hdf5"]["nodes"]

interesting = [
    n for n in nodes
    if any(term in n["path"].lower() for term in [
        "preferred",
        "combined",
        "/approximant",
        "/description",
        "/sampler/pe_algorithm",
        "/config_file/config/sampler",
        "/config_file/config/cosmology",
        "/config_file/config/default-prior",
        "/config_file/config/prior-dict",
    ])
]

pd.DataFrame(interesting)[["path", "dtype", "shape"]]

,path,dtype,shape
0,/C00:IMRPhenomXPHM-SpinTaylor/approximant,|S13,[1]
1,/C00:IMRPhenomXPHM-SpinTaylor/config_file/conf...,|S12,[1]
2,/C00:IMRPhenomXPHM-SpinTaylor/config_file/conf...,|S12,[1]
3,/C00:IMRPhenomXPHM-SpinTaylor/config_file/conf...,|S1163,[1]
4,/C00:IMRPhenomXPHM-SpinTaylor/config_file/conf...,|S7,[1]
5,/C00:IMRPhenomXPHM-SpinTaylor/config_file/conf...,|S145,[1]
6,/C00:IMRPhenomXPHM-SpinTaylor/description,|S20,[1]
7,/C00:IMRPhenomXPHM-SpinTaylor/meta_data/meta_d...,|S13,[1]
8,/C00:IMRPhenomXPHM-SpinTaylor/meta_data/meta_d...,NaN,NaN
9,/C00:IMRPhenomXPHM-SpinTaylor/meta_data/other/...,|S5,[1]


In [37]:
shallow_report["path"]

'/Projects/Data/macias/924994587/GW_Project_Data/raw/zenodo_20275769/IGWN-GWTC4p1-18965dda8_5-GW230627_015337-combined_PEDataRelease.hdf5'

In [38]:
import h5py

pe_path = Path(shallow_report["path"])

with h5py.File(pe_path, "r") as f:
    for label in [
        "C00:IMRPhenomXPHM-SpinTaylor",
        "C00:Mixed",
        "C00:SEOBNRv5PHM",
    ]:
        print("\n" + "=" * 70)
        print(label)

        paths = [
            "approximant",
            "description",
            "meta_data/other/preferred",
            "meta_data/sampler/pe_algorithm",
            "config_file/config/sampler",
            "config_file/config/cosmology",
            "config_file/config/default-prior",
        ]

        for subpath in paths:
            full = f"{label}/{subpath}"
            if full in f:
                value = f[full][()]
                print(f"{subpath:40s}: {value}")
            else:
                print(f"{subpath:40s}: <not present>")


C00:IMRPhenomXPHM-SpinTaylor
approximant                             : [b'IMRPhenomXPHM']
description                             : [b'No description found']
meta_data/other/preferred               : [b'False']
meta_data/sampler/pe_algorithm          : [b'bilby']
config_file/config/sampler              : [b'dynesty']
config_file/config/cosmology            : [b'Planck15_lal']
config_file/config/default-prior        : [b'BBHPriorDict']

C00:Mixed
approximant                             : <not present>
description                             : <not present>
meta_data/other/preferred               : <not present>
meta_data/sampler/pe_algorithm          : <not present>
config_file/config/sampler              : <not present>
config_file/config/cosmology            : <not present>
config_file/config/default-prior        : <not present>

C00:SEOBNRv5PHM
approximant                             : [b'SEOBNRv5PHM']
description                             : [b'No description found']
meta_data/oth

In [39]:
with h5py.File(pe_path, "r") as f:
    path = "C00:Mixed/meta_data/meta_data/combined"
    print(f[path][()])

[b'C00:IMRPhenomXPHM-SpinTaylor, C00:SEOBNRv5PHM']


In [40]:
with h5py.File(pe_path, "r") as f:
    labels = [
        "C00:IMRPhenomXPHM-SpinTaylor",
        "C00:SEOBNRv5PHM",
    ]

    prior_params = [
        "mass_1",
        "mass_2",
        "mass_ratio",
        "chirp_mass",
        "luminosity_distance",
        "a_1",
        "a_2",
        "tilt_1",
        "tilt_2",
    ]

    for label in labels:
        print("\n" + "=" * 70)
        print(label)

        for param in prior_params:
            path = f"{label}/priors/analytic/{param}"
            if path in f:
                print(f"{param:25s}: {f[path][()]}")
            else:
                print(f"{param:25s}: <not present>")


C00:IMRPhenomXPHM-SpinTaylor
mass_1                   : [b"Constraint(minimum=1, maximum=1000, name='mass_1', latex_label='$m_1$', unit=None)"]
mass_2                   : [b"Constraint(minimum=1, maximum=1000, name='mass_2', latex_label='$m_2$', unit=None)"]
mass_ratio               : [b"bilby.gw.prior.UniformInComponentsMassRatio(minimum=0.05, maximum=1.0, name='mass_ratio', latex_label='$q$', unit=None, boundary=None, equal_mass=False)"]
chirp_mass               : [b"bilby.gw.prior.UniformInComponentsChirpMass(minimum=6.38537095164235, maximum=6.4428834107576165, name='chirp_mass', latex_label='$\\\\mathcal{M}$', unit='$M_\\\\{\\\\{\\\\odot\\\\}\\\\}$', boundary=None)"]
luminosity_distance      : [b"bilby.gw.prior.UniformSourceFrame(minimum=50.0, maximum=471.599, cosmology='Planck15_LAL', name='luminosity_distance', latex_label='$d_L$', unit='Mpc', boundary=None)"]
a_1                      : [b"Uniform(minimum=0, maximum=0.99, name='a_1', latex_label='$a_1$', unit=None, boundary=Non

In [41]:
with h5py.File(pe_path, "r") as f:
    print(
        f["C00:Mixed/meta_data/meta_data/combined"][()]
    )

[b'C00:IMRPhenomXPHM-SpinTaylor, C00:SEOBNRv5PHM']


In [42]:
import numpy as np
with h5py.File(pe_path, "r") as f:
    label1 = "C00:IMRPhenomXPHM-SpinTaylor"
    label2 = "C00:SEOBNRv5PHM"

    p1 = f[f"{label1}/priors/analytic"]
    p2 = f[f"{label2}/priors/analytic"]

    names1 = set(p1.keys())
    names2 = set(p2.keys())

    print("Only in IMRPhenom:", sorted(names1 - names2))
    print("Only in SEOBNR:", sorted(names2 - names1))

    common = sorted(names1 & names2)

    differences = []
    for name in common:
        v1 = p1[name][()]
        v2 = p2[name][()]

        if not np.array_equal(v1, v2):
            differences.append((name, v1, v2))

    print("Number of common priors:", len(common))
    print("Number that differ:", len(differences))

    for item in differences:
        print("\n", item)

Only in IMRPhenom: []
Only in SEOBNR: []
Number of common priors: 77
Number that differ: 0


In [43]:
with h5py.File(pe_path, "r") as f:
    print(
        f["C00:Mixed/meta_data/meta_data/combined"][()]
    )

[b'C00:IMRPhenomXPHM-SpinTaylor, C00:SEOBNRv5PHM']


### Label decision for GW230627_015337

Candidate released PE labels:

- C00:IMRPhenomXPHM-SpinTaylor
- C00:SEOBNRv5PHM
- C00:Mixed

C00:Mixed is a combination of:
- C00:IMRPhenomXPHM-SpinTaylor
- C00:SEOBNRv5PHM

The two component analyses:
- use bilby with dynesty,
- use Planck15_lal cosmology,
- use BBHPriorDict,
- expose the required population coordinates,
- have identical stored analytic PE-prior definitions across all 77 prior entries.

The official GWOSC release designates the rerun C00:Mixed result as
the Default PE result for GW230627_015337.

Frozen label-selection rule:
Use the official release-designated Default PE result, independent of
sample count, ESS, or eventual population score.

Selected label for GW230627_015337:
C00:Mixed

## Required decision record

For each event, record: (1) every released candidate label, (2) the official population-analysis label/mixing policy, (3) available prior and configuration metadata, (4) sampled coordinates, (5) source-frame transformations and Jacobians, and (6) an independent numerical prior check. Only then update `config/pe_mapping.toml`. A tutorial waveform-priority example is not automatically the population-analysis policy.

In [31]:
## isolating first event
event_id = "GW230627_015337"

event_candidates = (
    candidates[candidates["event_id"] == event_id]
    .copy()
    .reset_index(drop=True)
)

event_candidates.T

,0,1,2
release,o4a,o4a,o4a
event_id,GW230627_015337,GW230627_015337,GW230627_015337
chi_eff_lower,0.037284,0.035924,0.034979
chi_eff_median,0.02522,0.022677,0.020598
chi_eff_upper,0.087738,0.08446,0.080986
chirp_mass_source_lower,0.071429,0.07445,0.07776
chirp_mass_source_median,6.014616,6.016959,6.019639
chirp_mass_source_upper,0.164317,0.167596,0.170742
gw_name,GW230627_015337,GW230627_015337,GW230627_015337
mass_1_source_lower,1.37363,1.315009,1.265286


In [32]:
event_candidates.columns.tolist()

['release',
 'event_id',
 'chi_eff_lower',
 'chi_eff_median',
 'chi_eff_upper',
 'chirp_mass_source_lower',
 'chirp_mass_source_median',
 'chirp_mass_source_upper',
 'gw_name',
 'mass_1_source_lower',
 'mass_1_source_median',
 'mass_1_source_upper',
 'mass_2_source_lower',
 'mass_2_source_median',
 'mass_2_source_upper',
 'network_matched_filter_snr_median',
 'redshift_lower',
 'redshift_median',
 'redshift_upper',
 'result_file_name',
 'result_label',
 'result_samples_key',
 'superevent_id']

In [7]:
event_candidates[
    [
        "release",
        "event_id",
        "result_samples_key",
        "result_file_name",
        "mass_1_source_median",
        "mass_2_source_median",
        "redshift_median",
    ]
]

,release,event_id,result_samples_key,result_file_name,mass_1_source_median,mass_2_source_median,redshift_median
0,o4a,GW230627_015337,C00:IMRPhenomXPHM-SpinTaylor,IGWN-GWTC4p1-18965dda8_5-GW230627_015337-combi...,8.483478,5.714881,0.066302
1,o4a,GW230627_015337,C00:Mixed,IGWN-GWTC4p1-18965dda8_5-GW230627_015337-combi...,8.449413,5.742252,0.065766
2,o4a,GW230627_015337,C00:SEOBNRv5PHM,IGWN-GWTC4p1-18965dda8_5-GW230627_015337-combi...,8.422318,5.762235,0.065132
